In [ ]:
# =========================================================
# 1. Load CBA list dataset
# 2. 
# =========================================================

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import re
import requests

# ---------- Paths ----------
# Define project directory
project_dir = Path("C:/Users/20234503/Desktop/Research/Strikes, Temperature, and Heat Safety Laws")
# Define raw CSV path
raw_csv = project_dir / "Raw Data" / "CBA" / "CBAList.csv"
# Define output directory for downloaded CBAs
output_dir = project_dir / "Raw Data" / "CBA" / "Agreements"

# ---------- Load ----------
# Specify encoding/dtypes/dates for deterministic loads across machines.
CBA = pd.read_csv(
    raw_csv,
    encoding="utf-8",
)

def extract_doc_ids(df, col="CBA File"):
    # Convert to string, pull the first run of digits, drop blanks, dedupe
    series = df[col].astype(str)
    ids = series.str.extract(r"(\d+)")[0]
    missing = ids.isna().sum()
    ids = ids.dropna().astype(int).astype(str)
    unique_ids = ids.unique().tolist()
    print(f"Found {len(unique_ids)} unique IDs; {missing} rows had no digits.")
    return unique_ids

doc_ids = extract_doc_ids(CBA)
doc_ids[:10]  # quick peek


ATTACHMENT_URL = "https://olmsapps.dol.gov/olpdr/GetAttachmentServlet?docId={doc_id}"

def sanitise_filename(name: str) -> str:
    # Remove/replace characters Windows can't use in filenames
    return re.sub(r'[\\/:*?"<>|]', "_", name).strip()

def filename_from_headers(doc_id: str, headers: dict) -> str:
    cd = headers.get("Content-Disposition", "") or headers.get("content-disposition", "")
    # Try RFC 5987 first: filename*=UTF-8''whatever.pdf
    m = re.search(r"filename\*\s*=\s*UTF-8''([^;]+)", cd, flags=re.IGNORECASE)
    if not m:
        # Fallback: filename="whatever.pdf" or filename=whatever.pdf
        m = re.search(r'filename\s*=\s*"?(?P<fn>[^";]+)"?', cd, flags=re.IGNORECASE)
    if m:
        return sanitise_filename(m.group(1 if m.lastindex is None else m.lastindex))
    return f"{doc_id}.pdf"

def download_one(doc_id: str, out_dir: Path, session: requests.Session, overwrite: bool = False) -> tuple[str, str]:
    """
    Returns (doc_id, status). status is 'ok', 'skipped', or an error message.
    """
    url = ATTACHMENT_URL.format(doc_id=doc_id)
    try:
        # HEAD first to get a clean filename without pulling the whole file if it already exists
        h = session.head(url, timeout=30, allow_redirects=True)
        if h.status_code != 200:
            return doc_id, f"HEAD {h.status_code}"
        fname = filename_from_headers(doc_id, h.headers)
        path = out_dir / fname
        if path.exists() and not overwrite:
            return doc_id, "skipped"

        # Now GET the content
        with session.get(url, stream=True, timeout=120, allow_redirects=True) as r:
            if r.status_code != 200:
                return doc_id, f"GET {r.status_code}"
            # Basic content-type sanity check
            ctype = (r.headers.get("Content-Type") or "").lower()
            if "pdf" not in ctype and "octet-stream" not in ctype:
                # Some files may still be PDFs; we proceed but note it
                pass
            tmp = path.with_suffix(path.suffix + ".part")
            with open(tmp, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 14):
                    if chunk:
                        f.write(chunk)
            tmp.replace(path)
        return doc_id, "ok"
    except requests.RequestException as e:
        return doc_id, f"network_error: {e}"

def download_all(doc_ids, out="cbas", cookie: str | None = None, referer: str | None = "https://olmsapps.dol.gov/olpdr/", overwrite=False):
    out_dir = Path(out)
    out_dir.mkdir(parents=True, exist_ok=True)

    headers = {"User-Agent": "Mozilla/5.0"}
    if referer:
        headers["Referer"] = referer
    if cookie:
        headers["Cookie"] = cookie

    results = []
    with requests.Session() as s:
        s.headers.update(headers)
        for i, doc_id in enumerate(doc_ids, start=1):
            status = download_one(str(doc_id), out_dir, s, overwrite=overwrite)
            results.append(status)
            print(f"[{i}/{len(doc_ids)}] {status[0]} -> {status[1]}")

    ok = sum(1 for _, st in results if st == "ok")
    skipped = sum(1 for _, st in results if st == "skipped")
    failed = [(d, st) for d, st in results if st not in ("ok", "skipped")]
    print(f"\nDone. Saved {ok}, skipped {skipped}, failed {len(failed)}. Files are in: {out_dir.resolve()}")
    if failed:
        print("Failures:")
        for d, st in failed[:20]:
            print(f"  {d}: {st}")
        if len(failed) > 20:
            print(f"  … and {len(failed) - 20} more")

# —— Run it ——
# Use the doc_ids list you created in Step 1:
# Example: download_all(doc_ids, out="cbas")
download_all(doc_ids, out=output_dir)


Found 2320 unique IDs; 0 rows had no digits.
[1/2320] 611 -> skipped
[2/2320] 655 -> ok
[3/2320] 689 -> ok
[4/2320] 1803 -> ok
[5/2320] 443 -> ok
[6/2320] 2647 -> ok
[7/2320] 1832 -> ok
[8/2320] 1804 -> ok
[9/2320] 565 -> ok
[10/2320] 639 -> network_error: HTTPSConnectionPool(host='olmsapps.dol.gov', port=443): Read timed out.
[11/2320] 748 -> ok
[12/2320] 2222 -> network_error: HTTPSConnectionPool(host='olmsapps.dol.gov', port=443): Read timed out.
[13/2320] 560 -> ok
[14/2320] 648 -> ok
[15/2320] 192 -> ok
[16/2320] 2247 -> ok
[17/2320] 1917 -> ok
[18/2320] 191 -> ok
[19/2320] 361 -> ok
[20/2320] 362 -> ok
[21/2320] 363 -> ok
[22/2320] 1506 -> ok
[23/2320] 1424 -> ok
[24/2320] 1381 -> ok
[25/2320] 1616 -> ok
[26/2320] 1357 -> ok
[27/2320] 1662 -> ok
[28/2320] 1688 -> ok
[29/2320] 1689 -> network_error: HTTPSConnectionPool(host='olmsapps.dol.gov', port=443): Read timed out.
[30/2320] 1776 -> ok
[31/2320] 901 -> ok
[32/2320] 1682 -> ok
[33/2320] 1683 -> skipped
[34/2320] 1332 -> ok
[35

KeyboardInterrupt: 